In [2]:
# =========================
# 1. IMPORT LIBRARIES
# =========================
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

# =========================
# 2. LOAD DATA
# =========================
df = pd.read_csv("C:/Users/Nilanjan Chanda/Downloads/tweet_sentiment/Tweets.csv")

# Keep only needed columns
df = df[['text', 'airline_sentiment']]

# =========================
# 3. NLP SETUP
# =========================
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
stop_words.discard('not')  # keep 'not' for sentiment

lemmatizer = WordNetLemmatizer()

# =========================
# 4. PREPROCESS FUNCTION
# =========================
def preprocess(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)

    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [lemmatizer.lemmatize(w) for w in words]

    return " ".join(words)

# Apply preprocessing
df['clean_text'] = df['text'].apply(preprocess)

print("\nSample cleaned data:")
print(df[['text', 'clean_text']].head())

# =========================
# 5. TF-IDF VECTORIZATION
# =========================
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X = vectorizer.fit_transform(df['clean_text'])

print("\nTF-IDF Shape:", X.shape)

# =========================
# 6. LABEL ENCODING
# =========================
encoder = LabelEncoder()
y = encoder.fit_transform(df['airline_sentiment'])

print("\nClasses:", encoder.classes_)

# =========================
# 7. TRAIN-TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 8. MODEL TRAINING
# =========================
model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

# =========================
# 9. EVALUATION
# =========================
y_pred = model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# =========================
# 10. TEST CUSTOM INPUT
# =========================
sample = ["worst flight ever delayed again"]

sample_clean = [preprocess(s) for s in sample]
sample_vec = vectorizer.transform(sample_clean)

prediction = model.predict(sample_vec)
print("\nCustom Input Prediction:", encoder.inverse_transform(prediction))

# =========================
# 11. MODEL INTERPRETATION
# =========================
# Handle sklearn version
try:
    feature_names = vectorizer.get_feature_names_out()
except:
    feature_names = vectorizer.get_feature_names()

for i, class_label in enumerate(encoder.classes_):
    coefs = model.coef_[i]

    top_indices = np.argsort(coefs)[-10:][::-1]
    top_words = [feature_names[j] for j in top_indices]

    print(f"\nTop words for class: {class_label}")
    print(top_words)

[nltk_data] Downloading package stopwords to C:\Users\Nilanjan
[nltk_data]     Chanda\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Nilanjan
[nltk_data]     Chanda\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!



Sample cleaned data:
                                                text  \
0                @VirginAmerica What @dhepburn said.   
1  @VirginAmerica plus you've added commercials t...   
2  @VirginAmerica I didn't today... Must mean I n...   
3  @VirginAmerica it's really aggressive to blast...   
4  @VirginAmerica and it's a really big bad thing...   

                                          clean_text  
0                                               said  
1       plus youve added commercial experience tacky  
2       didnt today must mean need take another trip  
3  really aggressive blast obnoxious entertainmen...  
4                               really big bad thing  

TF-IDF Shape: (14640, 5000)

Classes: ['negative' 'neutral' 'positive']

Accuracy: 0.8005464480874317

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.94      0.88      1889
           1       0.68      0.50      0.57       580
           2       0

In [4]:
from sklearn.decomposition import LatentDirichletAllocation

# Use same TF-IDF (or CountVectorizer optionally)
lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(X)

# Display topics
try:
    feature_names = vectorizer.get_feature_names_out()
except:
    feature_names = vectorizer.get_feature_names()

for i, topic in enumerate(lda.components_):
    top_words = [feature_names[j] for j in topic.argsort()[-10:]]
    print(f"\nTopic {i+1}:")
    print(top_words)
import numpy as np
df['engagement'] = np.random.randint(50, 500, size=len(df))
from sklearn.ensemble import RandomForestRegressor

X_train, X_test, y_train, y_test = train_test_split(X, df['engagement'], test_size=0.2)

reg = RandomForestRegressor()
reg.fit(X_train, y_train)


Topic 1:
['airline', 'lost', 'guy', 'much', 'late', 'luggage', 'not', 'bag', 'flight', 'thank']

Topic 2:
['thanks', 'time', 'flight', 'dm', 'sent', 'plane', 'hour', 'fleet fleek', 'fleet', 'fleek']

Topic 3:
['would', 'going', 'make', 'get', 'call', 'flight', 'trying', 'not', 'like', 'thanks']

Topic 4:
['time', 'thanks', 'get', 'not', 'im', 'dont', 'customer service', 'flight', 'service', 'customer']

Topic 5:
['flight cancelled', 'not', 'hour', 'help', 'hold', 'get', 'cancelled flightled', 'flightled', 'cancelled', 'flight']


RandomForestRegressor()

In [8]:
import pickle

pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))
pickle.dump(encoder, open("encoder.pkl", "wb"))
pickle.dump(lda, open("lda.pkl", "wb"))

print("Fresh models saved")

Fresh models saved


In [1]:
import os
print(os.getcwd())
print(os.listdir())

C:\Users\Nilanjan Chanda
['-1.14-windows.xml', '.appletviewer', '.azuredatastudio', '.cache', '.conda', '.condarc', '.config', '.continuum', '.eclipse', '.gitconfig', '.idlerc', '.ipynb_checkpoints', '.ipython', '.jupyter', '.matplotlib', '.ms-ad', '.node_repl_history', '.opera', '.oracle_jre_usage', '.p2', '.streamlit', '.VirtualBox', '.vscode', '3D Objects', 'advanced python modules.ipynb', 'anaconda3', 'app.py', 'AppData', 'Application Data', 'battery-report.html', 'book_scrape.ipynb', 'calender.html', 'Class_Object.ipynb', 'Contacts', 'Cookies', 'CrossDevice', 'CSV&PDF.ipynb', 'databases.py', 'Desktop', 'Documents', 'Downloads', 'eclipse', 'eclipse-workspace', 'email.ipynb', 'encoder.pkl', 'exception1.ipynb', 'exception2.ipynb', 'Favorites', 'Figure_1.png', 'file2.py', 'Flight+data.csv', 'img_prac.ipynb', 'index.html', 'IntelGraphicsProfiles', 'IPyWidgets.ipynb', 'IT1-Copy1.ipynb', 'IT1.ipynb', 'Kurt_Godel.jpg', 'lda.pkl', 'Links', 'Local Settings', 'MicrosoftEdgeBackups', 'model.p

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,3),
    min_df=2
)
X = vectorizer.fit_transform(df['clean_text'])

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(X)

# Display topics
try:
    feature_names = vectorizer.get_feature_names_out()
except:
    feature_names = vectorizer.get_feature_names()

for i, topic in enumerate(lda.components_):
    top_words = [feature_names[j] for j in topic.argsort()[-10:]]
    
    print(f"\nTopic {i+1}:")
    print(top_words)

In [ ]:
import os

files = [
    "model.pkl",
    "vectorizer.pkl",
    "encoder.pkl",
    "lda.pkl"
]

for f in files:
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted {f}")

In [ ]:
def engagement_score(text):

    score = 50

    # hashtags
    score += text.count("#") * 8

    # mentions
    score += text.count("@") * 5

    # exclamation
    score += text.count("!") * 4

    # questions
    score += text.count("?") * 6

    # length bonus
    length = len(text.split())

    if 8 <= length <= 20:
        score += 15

    # emotional / viral words
    strong_words = [
        "awesome", "amazing", "breaking",
        "love", "future", "best",
        "shocking", "revolution"
    ]

    for word in strong_words:
        if word in text.lower():
            score += 8

    return min(score, 100)

In [ ]:
test_post = "Awesome AI breakthrough! What do you think? #AI"

print("Engagement Score:",
      engagement_score(test_post))

In [ ]:
def predict_engagement(text):

    score = 0
    reasons = []

    # Length
    words = len(text.split())

    if 8 <= words <= 25:
        score += 20
        reasons.append("Good post length")
    else:
        reasons.append("Length can be improved")

    # Hashtags
    hashtag_count = text.count("#")

    if hashtag_count > 0:
        score += 20
        reasons.append("Uses hashtags")
    else:
        reasons.append("No hashtags")

    # Questions
    if "?" in text:
        score += 15
        reasons.append("Encourages interaction")
    else:
        reasons.append("No engagement question")

    # Emotional punctuation
    if "!" in text:
        score += 10
        reasons.append("Strong emotional tone")

    # Strong words
    strong_words = [
        "amazing", "awesome", "future",
        "revolution", "best", "shocking",
        "massive", "incredible"
    ]

    for word in strong_words:
        if word in text.lower():
            score += 10
            reasons.append(f"Contains strong word: {word}")
            break

    # Final category
    if score >= 50:
        level = "High"
    elif score >= 25:
        level = "Medium"
    else:
        level = "Low"

    return level, reasons


In [ ]:
post = """
AI is revolutionizing healthcare 🚀
Will doctors be replaced? #AI #Future
"""

level, reasons = predict_engagement(post)

print("Engagement:", level)

for r in reasons:
    print("-", r)


In [5]:
print(vectorizer.max_features)

5000


In [6]:
print(X.shape)

(14640, 5000)


In [7]:
print(model.n_features_in_)
print(lda.n_features_in_)
print(len(vectorizer.get_feature_names()))

5000
5000
5000
